> Historical development experiment. Read [notebook20](20_practical_options_report.ipynb) for the final recommendation.

# Full-clip test: better context checks, unresolved animal trigger

**Do not promote yet.** The later beach segment still opens the animal gate on ambiguous/contradictory evidence. Verifying context boxes cannot repair an incorrect animal trigger.

**141 aerial frames through all four complete source clips**, sampled at2FPS, plus **24 ground-release frames** as a retention regression. These are previously inspected development sources.

| Clip | Person frames: raw / exclusive / class-specific | Vehicle frames: raw / exclusive / class-specific |
|---|---:|---:|
| Platform | 0 / 0 / 0 | 0 / 0 / 0 |
| Surf | 7 / 0 / 0 | 0 / 0 / 0 |
| Rocks | 1 / 0 / 0 | 15 / 0 / 0 |
| Beach control | 5 / 1 / 1 | 6 / 6 / 6 |
| Ground release | 19 / 19 / 19 | 15 / 9 / 15 |

Raw = animal-gated detector proposals. Exclusive = the original single-class crop verifier. Class-specific = asks whether the proposed person or vehicle is actually supported inside the target rectangle, allowing other objects to coexist. The table holds animal decisions fixed. **Counts are display/proposal frames, not accuracy, recall or unique-object counts.**

## What the longer test exposed

- The surf and rock context proposals were reviewed across all eligible sampled frames, not just the earlier examples.
- The exclusive classifier sometimes called a truck target “person” because responders overlapped the truck. Class-specific presence is a better formulation for overlapping objects; its full-task A/B uses exactly the same crop bytes.
- At approximately15seconds of the beach clip, the crop reviewer called a tiny shape a quadruped, while the whole-scene reviewer justified a mammal “yes” with an explanation mentioning birds. Independent visual review cannot reliably identify a mammal in that tiny crop. The gate opened six frames; the prior first12seconds-only result did not cover this failure.
- No verified aerial marine-animal-plus-human/vehicle positive is available here. The ground-release retention result cannot fill that gap.

## Evidence clearing and scene changes

The constructed hard cut left raw context visible in 6 control frames with the old hold, versus 0 with dual-negative clearing and 0 with a supplied scene reset. This uses completed model decisions at source time, not live automatic cut detection.
Negative clearing reduced surf animal-evidence frames from20 to19, and did not stop the later beach trigger because both reviewers answered positive there. Clearing can also drop context during real animal occlusion. Natural exit timing remains unannotated.

## Calls, latency and verification

New requests: **20 animal + 57 exclusive-context + 57 class-specific-context = 134**. Failed new requests across phases: 0. Original32 aerial animal responses were reused.
Class-specific context requests took a median **2.42s** and maximum **6.52s**, excluding queue wait. No context approval is propagated across frames. This is offline replay; live2FPS has not been demonstrated.
Source/frame/request-image/weight/code hashes, full2FPS grid coverage, per-region mappings and policy replay were audited. The ground reference was restricted by exact source-frame hashes to its24 release samples; correction lineage is archived. Logic fixtures are separate from model performance.

## Reproduce and limits

Run context_video_infer_v1.py, context_video_review_v1.py, context_ground_scope_correction_v1.py, context_video_archive_v1.py, context_hypothesis_review_v1.py and context_video_finalize_v1.py in their documented environments. Frozen plans/results are under research/aerial_detector/results/context-video-v1; media and exact request crops are under ~/aerial-data/context-video-v1. Notebook cells only display cached artifacts.
Both reviewers share a Gemini model and can make correlated mistakes. Source altitude/equipment/pretraining overlap, exhaustive boxes, identities, natural animal visibility and real-time buffering are not validated. No training, production integration, flight control, automatic incident/dispatch or deployment.

## Sources

- [raft](https://www.pexels.com/video/aerial-view-of-seals-resting-on-raft-40906785/) — Marc Espejo/Pexels; complete 18.35s source.
- [surf](https://www.pexels.com/video/a-sea-lion-swimming-along-the-waves-in-a-beach-3006844/) — Athena Sandrini/Pexels; complete 15.04s source.
- [rocks](https://www.pexels.com/video/aerial-view-of-seals-resting-on-ocean-rocks-30824194/) — Lost Souls NV/Pexels; complete 17.38s source.
- [crowded_beach](https://www.pexels.com/video/aerial-view-of-beach-with-people-and-umbrellas-32726982/) — 8 K/Pexels; complete 18.94s source.
- Ground retention: NOAA Fisheries/Hawaii Marine Animal Response R7AF release, same segment as study17.

![Research evidence](../assets/context-verifier-comparison.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

In [ ]:
from pathlib import Path
import json
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'research/aerial_detector/results').is_dir())
research=root/'research/aerial_detector'
print(json.dumps(json.loads((research/'results/context-video-v1/report.json').read_text()),indent=2))